In [ ]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

# Install Necessary Packages

In [ ]:
!git clone https://github.com/FlagOpen/FlagEmbedding.git
!pip install -e .

In [ ]:
!pip install huggingface_hub
!pip install datasets
!pip install transformers
!pip install loguru -qU
!pip install tokenizers
!pip install langchain -qU
!pip install bitsandbytes -qU
!pip install accelerate==0.21.0
!pip install peft==0.4.0
!pip install trl==0.4.7
!pip install guardrail-ml==0.0.12
!pip install flash-attn --no-build-isolation
!pip install -U FlagEmbedding

In [ ]:
class PromptTemplate:
      system_prompt = None


      def __init__(self, system_prompt=None):
          self.system_prompt = system_prompt
          self.user_messages = []
          self.model_replies = []

      def add_user_message(self, message: str, return_prompt=True):
          self.user_messages.append(message)
          if return_prompt:
              return self.build_prompt()

      def add_model_reply(self, reply: str, includes_history=True, return_reply=True):
          reply_ = reply.replace(self.build_prompt(), "") if includes_history else reply
          self.model_replies.append(reply_)
          if len(self.user_messages) != len(self.model_replies):
              raise ValueError(
                  "Number of user messages does not equal number of system replies."
              )
          if return_reply:
              return reply_

      def get_user_messages(self, strip=True):
          return [x.strip() for x in self.user_messages] if strip else self.user_messages

      def get_model_replies(self, strip=True):
          return [x.strip() for x in self.model_replies] if strip else self.model_replies

      def build_prompt(self):
          if len(self.user_messages) != len(self.model_replies) + 1:
              raise ValueError(
                  "Error: Expected len(user_messages) = len(model_replies) + 1. Add a new user message!"
              )

          if self.system_prompt is not None:
              SYS = f"[INST] <<SYS>>\n{self.system_prompt}\n<</SYS>>"
          else:
              SYS = ""

          CONVO = ""
          SYS = "<s>" + SYS
          for i in range(len(self.user_messages) - 1):
              user_message, model_reply = self.user_messages[i], self.model_replies[i]
              conversation_ = f"{user_message} [/INST] {model_reply} </s>"
              if i != 0:
                  conversation_ = "[INST] " + conversation_
              CONVO += conversation_

          CONVO += f"[INST] {self.user_messages[-1]} [/INST]"

          return SYS + CONVO

# Import Necessary Packages

In [ ]:
import os
from glob import glob
import pandas as pd
import json
import time
import requests
import random
from loguru import logger
import re
#from huggingface_hub import HfApi, HfFolder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np


In [ ]:
from transformers import(AutoTokenizer,
                         AutoModelForMultipleChoice,
                         AutoModelForCausalLM,
                         AutoTokenizer,

                         GenerationConfig,
                         BitsAndBytesConfig,

                         pipeline,
                         Conversation,
                         logging,
                         )
from datasets import load_dataset
from tokenizers import Tokenizer

import warnings
warnings.filterwarnings("ignore")


### HELPER FUNCTION for Multi-Agents Debate ###

In [ ]:
# this part is used to gen_mmlu

def construct_message(agents, question, idx):
    if len(agents) == 0:
        return {"role": "user", "content": "Can you double check that your answer is correct. Put your final answer in the form (X) at the end of your response."}

    prefix_string = "These are the solutions to the problem from other agents: "

    for agent in agents:
        agent_response = agent[idx]["content"]
        response = "\n\n One agent solution: ```{}```".format(agent_response)

        prefix_string = prefix_string + response

    prefix_string = prefix_string + """\n\n Using the reasoning from other agents as additional advice, can you give an updated answer? Examine your solution and that other agents step by step. /n/n Here is the original question: {}. """.format(question)
    return {"role": "user", "content": prefix_string}


def construct_assistant_message(completion):
    # just construct the assistant_message directly.

    return {"role": "assistant", "content": completion}


def generate_answer(answer_context):
    """
    input: list of dict, answer_context
    output: str, content
    """
    try:
        # Generate a prompt
        messages = answer_context

        tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")

        # Generate a response
        outputs = base_model.generate(
                                tokenized_chat,
                                max_new_tokens=400,
                                max_time=90, # control the generation time

                                do_sample = True,

                                top_k = 50, # both top_k and top_p combined to help me control the quality of logit
                                top_p = 0.9,

                                temperature= 0.1,
                                #num_return_sequences= 1, # control the num of returned sequence, to less the recall api time

                                repetition_penalty= 1.5,
                                )

        # parse output_text
        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Removing the query content and everything before it
        # Find the index where the query content ends in the output_text
        end_idx_of_query = output_text.find(messages[0]['content']) + len(messages[0]['content'])
        cleaned_output_text = output_text[end_idx_of_query:].strip()

        # Further clean up if needed
        pattern = r"\[.*?\]|\(.*?\)|\{.*?\}"
        cleaned_output_text = re.sub(pattern, "", cleaned_output_text)
    except:
        print("retrying due to an error......")
        time.sleep(20)
        return generate_answer(answer_context)

    return  cleaned_output_text


def parse_question_answer(df, ix):
    question = df.iloc[ix, 0]
    a = df.iloc[ix, 1]
    b = df.iloc[ix, 2]
    c = df.iloc[ix, 3]
    d = df.iloc[ix, 4]

    question = "Can you answer the following question as accurately as possible? {}: /n/n A) {}, /n B) {}, /n C) {}, /n D) {}. /n Explain your answer, putting the answer in the form (X) at the end of your response.".format(question, a, b, c, d)

    answer = df.iloc[ix, 5]

    return question, answer



# an alternative way to implement generating the text answer
def sample_model(prompt):
    conversation_pipeline = pipeline(
                                    'conversational',
                                    model=model,
                                    tokenizer=tokenizer,
                                    max_new_tokens=300,
                                    max_time=90, # control the generation time

                                    do_sample = True,

                                    top_k = 75, # both top_k and top_p combined to help me control the quality of logit
                                    top_p = 0.9,

                                    temperature= 0.9,
                                    #num_return_sequences= 1, # control the num of returned sequence, to less the recall api time

                                    repetition_penalty= 1.2,
                                    eos_token_id= tokenizer.eos_token_id,
                                    pad_token_id= tokenizer.eos_token_id,
                                    bos_token_id= tokenizer.eos_token_id,
                                    )
    conversation = Conversation(prompt)
    conversation_pipeline([conversation])
    return conversation.generated_responses[-1]


# Helper function for printing docs

def pretty_print_docs(docs):
    print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]))



# 1. Set up the model

In [ ]:

#base_model_path = '/content/drive/MyDrive/Hallucination/Llama2_7b_base/Llama2_7b_Finance_FT_3/Llama2-7b_Finance_FT_3_with_lora'
base_model_path = 'NousResearch/Llama-2-7b-hf'


# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('NousResearch/Llama-2-7b-hf',
                                          use_fast=True,
                                          #chat_template = set_template
                                          )


# Load the trained model
base_model = AutoModelForCausalLM.from_pretrained(base_model_path,
                                             #quantization_config=bnb_config,
                                             trust_remote_code=True,
                                             load_in_8bit=True,
                                             device_map="auto",
                                             #use_flash_attention_2=True,
                                             )

base_model.config.use_cache = False # Because, we just take the performance of single turn into consideration,

#model.push_to_hub("Llama2-7b_Finance_lora_3")

# If you're using a GPU, move the model to GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#base_llm = model#.to(device) # int8,int can not put into .to()


In [ ]:
# Print the BOS and EOS tokens
print("BOS Token:", tokenizer.bos_token)
print("EOS Token:", tokenizer.eos_token)
print("PAD Token:", tokenizer.pad_token)
print("SEP Token:", tokenizer.sep_token)
print("MASK Token:", tokenizer.mask_token)

In [ ]:
print("BOS Token id:", tokenizer.bos_token_id)
print("EOS Token id:", tokenizer.eos_token_id)
print("PAD Token id:", tokenizer.pad_token_id)
print("SEP Token id:", tokenizer.sep_token_id)
print("MASK Token id:", tokenizer.mask_token_id)

In [ ]:
tokenizer.default_chat_template

# 2. Retrieval Augmention Generation(RAG)

In [ ]:
!pip install yfinance
!pip install wikipedia
!pip install faiss-GPU

In [ ]:
from langchain.agents import AgentType, initialize_agent


#tools
from langchain.tools.yahoo_finance_news import YahooFinanceNewsTool

#retrievers
from langchain.retrievers import WikipediaRetriever

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

from langchain.retrievers.document_compressors import EmbeddingsFilter

from FlagEmbedding import (FlagReranker, FlagModel, LLMEmbedder)



# 2.1 tools \[   YahooFinanceNewsTool,\]

In [ ]:
# this tool-- YahooFinanceNewsTool only be used for financial test
tools = [YahooFinanceNewsTool()]

# 2.2 wikipedia_retirevers as a import documents source

In [ ]:
# because we are dealing with MMLU problem, which is discrimination evaluation, so wikipediaretriever
wiki_retriever = WikipediaRetriever()

In [ ]:
query_wiki = 'kobe bryant'

In [ ]:
documents = wiki_retriever.get_relevant_documents(query = query_wiki)
documents

In [ ]:
documents = "\n".join([f"Document {i}: {doc.page_content}" for i, doc in enumerate(docs)])

In [ ]:
documents

# 2.3 Embedding Model, Reranker, ContextualCompressionRetriever

In [ ]:
reranker = FlagReranker('BAAI/bge-reranker-large', use_fp16= True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

score = reranker.compute_score(['query', 'passage'])
print(score)

scores = reranker.compute_score([['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']])
print(scores)

In [ ]:
from langchain.embeddings import HuggingFaceBgeEmbeddings
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embed_model = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
    query_instruction="Generate a representation for this sentence to retrieve relevant articles: "
)
embed_model.query_instruction = "Generate a representation for this sentence to retrieve relevant articles:"

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain.vectorstores import FAISS

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

texts = text_splitter.split_documents(documents)
base_retriever = FAISS.from_documents(texts, embed_model).as_retriever()

docs = base_retriever.get_relevant_documents("Kobe is great.")
pretty_print_docs(docs)

In [ ]:
from langchain.retrievers.document_compressors import EmbeddingsFilter

embeddings = embed_model
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.5)
compression_retriever = ContextualCompressionRetriever(base_compressor=embeddings_filter, base_retriever= base_retriever)

compressed_docs = compression_retriever.get_relevant_documents("who is kobe bryant?")


In [ ]:
pretty_print_docs(compressed_docs)

# 2.5 Build up the RAG part

In [ ]:



def RAG(answer_context):

    # query for retirever
    messages = answer_context
    query = messages[0]['content']


    docs = wiki_retriever.get_relevant






    return documents

# Generation on MMLU data

In [ ]:
agents = 2
rounds = 2

tasks = glob("/content/drive/MyDrive/Hallucination/Parse_Data/data/test/*.csv")

dfs = [pd.read_csv(task) for task in tasks]

random.seed(123)
response_dict = {}

for i in range(100):
    df = random.choice(dfs)
    ix = len(df)
    idx = random.randint(0, ix-1)

    question, answer = parse_question_answer(df, idx)

    agent_contexts = [[{"role": "user", "content": question}] for agent in range(agents)]

    # initilized debate rounds
    for round in range(rounds):
        for i, agent_context in enumerate(agent_contexts):

            if round != 0:
                agent_contexts_other = agent_contexts[:i] + agent_contexts[i+1:]
                message = construct_message(agent_contexts_other, question, 2 * round - 1)
                agent_context.append(message)

            completion = generate_answer(agent_context)

            assistant_message = construct_assistant_message(completion)
            agent_context.append(assistant_message)
            print(completion)

    # RAG part provides relevant documents



    response_dict[question] = (agent_contexts, answer)


json.dump(response_dict, open("mmlu_{}_{}.json".format(agents, rounds), "w"))

#wandb.finish()

In [ ]:
file_path = 'mmlu_2_2.json'

# Open and read the JSON file
with open(file_path, 'r') as file:
    data = json.load(file)

# Now 'data' contains the contents of the JSON file as a Python dictionary
# You can process or print the data as needed
print(data)

In [ ]:
len(data['Can you answer the following question as accurately as possible? Let T: R^2 -> R^2 be the linear transformation that maps the point (1, 2) to (2, 3) and the point (-1, 2) to (2, -3). Then T maps the point (2, 1) to: /n/n A) (1, 6), /n B) (-1, 4), /n C) (3, 2), /n D) (-4, 3). /n Explain your answer, putting the answer in the form (X) at the end of your response.'
])

In [ ]:
len(data['Can you answer the following question as accurately as possible? Let T: R^2 -> R^2 be the linear transformation that maps the point (1, 2) to (2, 3) and the point (-1, 2) to (2, -3). Then T maps the point (2, 1) to: /n/n A) (1, 6), /n B) (-1, 4), /n C) (3, 2), /n D) (-4, 3). /n Explain your answer, putting the answer in the form (X) at the end of your response.'
][0])

In [ ]:
len(data['Can you answer the following question as accurately as possible? Let T: R^2 -> R^2 be the linear transformation that maps the point (1, 2) to (2, 3) and the point (-1, 2) to (2, -3). Then T maps the point (2, 1) to: /n/n A) (1, 6), /n B) (-1, 4), /n C) (3, 2), /n D) (-4, 3). /n Explain your answer, putting the answer in the form (X) at the end of your response.'
][0][0])

In [ ]:
data['Can you answer the following question as accurately as possible? Let T: R^2 -> R^2 be the linear transformation that maps the point (1, 2) to (2, 3) and the point (-1, 2) to (2, -3). Then T maps the point (2, 1) to: /n/n A) (1, 6), /n B) (-1, 4), /n C) (3, 2), /n D) (-4, 3). /n Explain your answer, putting the answer in the form (X) at the end of your response.'
][0][0]

In [ ]:
from google.colab import files

files.download(file_path)

# Evaluation on MMLU test data

# Break_Down Experiment

In [ ]:

agents = 2
rounds = 2

tasks = glob("/content/drive/MyDrive/Hallucination/Parse_Data/data/test/*.csv")

dfs = [pd.read_csv(task) for task in tasks]

random.seed(3)
response_dict = {}

for i in range(2):
    df = random.choice(dfs)
    ix = len(df)
    idx = random.randint(0, ix-1)

    question, answer = parse_question_answer(df, idx)

    agent_contexts = [[{"role": "user", "content": question}] for agent in range(agents)]

In [ ]:
agent_contexts

In [ ]:
query = [{'role': 'user',
   'content': "<s> Can you answer the following question as accurately as possible?  Patients with which of the following diseases are treated with injections of vitamin B-12?:  /n/n A) Bell's palsy, /n B) Crohn's disease, /n C) Pernicious anemia, /n D) Graves' disease. /n Explain your answer, putting the answer in the form (X) at the end of your response. </s>"}]

In [ ]:
question = query[0]['content']
question

In [ ]:
output_text = generate_answer(query)

In [ ]:
output_text

In [ ]:
query_1 = [{
            'role': 'user',
            'content': "<s> can you tell me who is kobe bryant? </s>"
}]

In [ ]:
output_text1 = generate_answer(query_1)
output_text1

In [ ]:
completion

In [ ]:
assistant_message = construct_assistant_message(completion)

In [ ]:
assistant_message

In [ ]:
output_text